# Task 4: TCN with 36 steps + time of day + day of week

The one input combination not tested yet. Same setup as the other task-4 runs: TCN with 32 channels and kernel 2,
6 layers (needed to see 36 steps back), 10 epochs, learning rate 0.001, seed 42. Result: `results_in36.zip` (Output tab),
to unzip in the repo folder; the task-4 cell of the notebook then shows it as `speed+tod+dow_in36`.

**Setup on Kaggle:** Add Input -> the same dataset as for the MLP; Settings -> Accelerator: **GPU T4 x2**, Internet: **on**.
Then **Save Version -> Save & Run All (Commit)**.


In [ ]:
import glob, json, os, shutil, subprocess, sys, time, zipfile

# the uploaded code + data + tuning database, wherever Kaggle put it; /kaggle/input is read-only, so work on a copy
WORK = '/kaggle/working/topic4'
marker = glob.glob('/kaggle/input/**/src/utils/tuning/nn_tuning.py', recursive=True)
if marker:                                   # Kaggle unpacked the zip
    shutil.copytree(marker[0].removesuffix('/src/utils/tuning/nn_tuning.py'), WORK, dirs_exist_ok=True)
else:                                        # the zip was kept as a file: unpack it here
    zipfile.ZipFile(glob.glob('/kaggle/input/**/*.zip', recursive=True)[0]).extractall('/kaggle/working')
os.chdir(WORK)
print(sorted(os.listdir('.')), '| tuning database found:', os.path.exists('checkpoints/tuning/optuna.db'))

In [ ]:
# BasicTS without its pinned extras (its old `transformers` does not build on Kaggle and our code does not use it),
# plus the few small packages our code imports that Kaggle does not have
!pip install -q --no-deps basicts==1.1.0 easy-torch==1.3.3
!pip install -q easydict setproctitle optuna
!python -c "import basicts, easytorch, optuna; from src.utils.tuning.nn_tuning import MODELS; print('BasicTS', basicts.__version__, '| models:', list(MODELS))"
import torch
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2'
print(torch.__version__, torch.cuda.device_count(), 'x', torch.cuda.get_device_name(0))

## Train (about 20-30 minutes on the GPU)


In [ ]:
!python -c "from basicts import BasicTSLauncher; from src.model.baseline_temporal_models.tcn import TCN, TCNConfig; from src.utils.config import make_cfg; BasicTSLauncher.launch_training(make_cfg(TCN, TCNConfig(input_len=36, num_layers=6, use_tod=True, use_dow=True), input_len=36, num_epochs=10, use_timestamps=True, ckpt_save_dir='checkpoints/features/TCN_speed+tod+dow_in36'))" > checkpoints/in36.log 2>&1
print(open('checkpoints/in36.log').read()[-1500:])          # the end of the log: test results


## Result and download


In [ ]:
print(json.dumps(json.load(open(glob.glob('checkpoints/features/TCN_speed+tod+dow_in36/*/test_metrics.json')[0])), indent=1))
with zipfile.ZipFile('/kaggle/working/results_in36.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob('checkpoints/features/**', recursive=True):
        if os.path.isfile(path) and path.endswith(('test_metrics.json', 'cfg.json', '.log', '_best_val_MAE.pt')):
            z.write(path)
print('results_in36.zip -> Output tab -> download, then unzip it in the repo folder on the PC')
